[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C50_HuggingFace_Ecosystem_Course/02_tokenizers_datasets/02_tokenizers_datasets.ipynb)

# 02 · tokenizers 与 datasets（迷你复刻 offsets / word_ids / map / streaming / collator）

目标：把 **fast tokenizer 的 offset 与 word_ids → padding/truncation 的四种组合 → Dataset.map 的批处理与指纹缓存 →
IterableDataset 与缓冲打乱 → DataCollator 的 -100 构造** 从零写一遍。

路线：迷你 fast tokenizer（带 offsets）→ 子词到词的对齐 → span 用 offsets 映射回原文 →
padding 浪费与按长度分组 → 截断率统计 → map 的批处理与指纹 → 缓冲打乱 →
collator 与 -100 → 「打印一个 batch」检查器 → ✏️ 练习 → 📖 答案 → 🧪 浪费率胶囊。

> 心智模型：**数据层的坑几乎全是「形状与对齐」的坑**——全都不报错，只让效果变差。

## 1 · 迷你 fast tokenizer：offsets 与 word_ids

`offset_mapping` 让你把 token 级预测映射回**原始字符串的字符位置**；
`word_ids()` 让你把**词级标注**对齐到子词级。这两个是 fast tokenizer 真正不可替代的地方。

In [ ]:
import numpy as np, math, re, json, hashlib, random
rng = np.random.default_rng(0)

CLS, SEP, UNK, PAD = '[CLS]', '[SEP]', '[UNK]', '[PAD]'
# 一个迷你 WordPiece 风格词表（含 ## 续接片）
VOCAB = [PAD, CLS, SEP, UNK, 'wash', '##ing', '##ton', 'is', 'nice', 'new', 'york',
         'city', 'the', 'capital', 'of', 'usa', '.', 'very', 'good']
STOI = {w: i for i, w in enumerate(VOCAB)}

class MiniFastTokenizer:
    '''迷你 fast tokenizer：返回 input_ids / attention_mask / offset_mapping / word_ids。'''
    def __init__(self, vocab):
        self.stoi = {w: i for i, w in enumerate(vocab)}
        self.itos = {i: w for w, i in self.stoi.items()}
        # 按长度降序，贪心最长匹配（WordPiece 的经典做法）
        self.pieces = sorted([w for w in vocab if not w.startswith('[')],
                             key=len, reverse=True)

    def _split_words(self, text):
        '''返回 [(词, 起始字符, 结束字符)]，标点独立成词。'''
        out = []
        for m in re.finditer(r"\w+|[^\w\s]", text):
            out.append((m.group(0).lower(), m.start(), m.end()))
        return out

    def _wordpiece(self, word, w_start):
        '''把一个词切成子词，返回 [(piece, start_char, end_char)]。'''
        pieces, i = [], 0
        while i < len(word):
            matched = None
            for L in range(len(word) - i, 0, -1):
                cand = word[i:i + L]
                key = cand if i == 0 else '##' + cand
                if key in self.stoi:
                    matched = (key, w_start + i, w_start + i + L); i += L; break
            if matched is None:
                return [(UNK, w_start, w_start + len(word))]
            pieces.append(matched)
        return pieces

    def __call__(self, text, max_length=None, truncation=False,
                 return_offsets_mapping=True, add_special_tokens=True):
        words = self._split_words(text)
        ids, offs, wids = [], [], []
        if add_special_tokens:
            ids.append(self.stoi[CLS]); offs.append((0, 0)); wids.append(None)
        for wi, (w, s, e) in enumerate(words):
            for piece, ps, pe in self._wordpiece(w, s):
                ids.append(self.stoi.get(piece, self.stoi[UNK]))
                offs.append((ps, pe)); wids.append(wi)
        if add_special_tokens:
            ids.append(self.stoi[SEP]); offs.append((0, 0)); wids.append(None)
        if truncation and max_length and len(ids) > max_length:
            keep = max_length - (1 if add_special_tokens else 0)
            ids, offs, wids = ids[:keep], offs[:keep], wids[:keep]
            if add_special_tokens:
                ids.append(self.stoi[SEP]); offs.append((0, 0)); wids.append(None)
        enc = {'input_ids': ids, 'attention_mask': [1] * len(ids)}
        if return_offsets_mapping:
            enc['offset_mapping'] = offs
        enc['_word_ids'] = wids
        return enc

    def tokens(self, ids): return [self.itos[i] for i in ids]

tok = MiniFastTokenizer(VOCAB)
TEXT = 'Washington is nice'
enc = tok(TEXT)
print(f'{"token":<8s} {"id":>4s} {"offset":>10s} {"word_id":>8s} {"原文切片":>10s}')
for t, i, o, w in zip(tok.tokens(enc['input_ids']), enc['input_ids'],
                      enc['offset_mapping'], enc['_word_ids']):
    slice_ = TEXT[o[0]:o[1]] if o != (0, 0) else ''
    print(f'{t:<8s} {i:>4d} {str(o):>10s} {str(w):>8s} {slice_!r:>12s}')

assert tok.tokens(enc['input_ids'])[1:4] == ['wash', '##ing', '##ton']
assert enc['_word_ids'][1:4] == [0, 0, 0], '三个子词都属于词 0'
assert enc['_word_ids'][0] is None and enc['_word_ids'][-1] is None, '特殊 token 的 word_id 是 None'
# 关键：把词 0 的三个子词的 offset 拼起来 = 原文的完整词
w0 = [o for o, w in zip(enc['offset_mapping'], enc['_word_ids']) if w == 0]
assert TEXT[w0[0][0]:w0[-1][1]] == 'Washington', '子词 offset 拼起来应还原原词'
print(f"\n✅ 词 0 的 offset 区间 [{w0[0][0]}, {w0[-1][1]}) -> {TEXT[w0[0][0]:w0[-1][1]]!r}")
print('   注意特殊 token 的 offset 是 (0,0) 而**不是 None** —— 用 word_ids 的 None 来识别它们。')

### 用途 ①：词级标注 → 子词级 labels（C49 模块 03 那个坑）

In [ ]:
LABELS = ['O', 'B-LOC', 'I-LOC', 'B-PER', 'I-PER']
L2I = {l: i for i, l in enumerate(LABELS)}

def align_word_labels(word_ids, word_labels, label_all_subwords=False):
    '''每个词的**第一个**子词取标签，其余填 -100（除非 label_all_subwords）。'''
    out, prev = [], None
    for wid in word_ids:
        if wid is None:
            out.append(-100)
        elif wid != prev:
            out.append(word_labels[wid])
        else:
            if label_all_subwords:
                lab = LABELS[word_labels[wid]]
                out.append(L2I['I-' + lab[2:]] if lab.startswith('B-') else word_labels[wid])
            else:
                out.append(-100)
        prev = wid
    return out

word_labels = [L2I['B-LOC'], L2I['O'], L2I['O']]     # Washington / is / nice
labs = align_word_labels(enc['_word_ids'], word_labels)
print('word_ids:', enc['_word_ids'])
print('labels  :', labs, ' (-100 = 不计损失)')
assert labs == [-100, L2I['B-LOC'], -100, -100, L2I['O'], L2I['O'], -100]
n_signal = sum(1 for x in labs if x != -100)
assert n_signal == len(word_labels), '每个词只贡献一个训练信号'
print(f'✅ {len(enc["input_ids"])} 个 token 里只有 {n_signal} 个产生损失（= 词数）')

# 反面：不做对齐，直接把词级标签按位置铺开
naive = word_labels + [0] * (len(enc['input_ids']) - len(word_labels))
print(f'\n❌ 不做对齐: {naive}')
print('   -> 模型会学到「把 Washington 的第 2、3 个子词标成 O」，实体级 F1 莫名很低')
assert naive != labs
print('✅ 这就是 word_ids() 不可替代的原因 —— 手工按空格切在真实数据上极易出错')

### 用途 ②：span 预测 → 用 offsets 映射回原文字符

**不能用 decode 拼答案**——分词不是无损可逆的（空格规范化、lowercase、Unicode 归一化）。

In [ ]:
TEXT2 = 'New York City is the capital of USA.'
enc2 = tok(TEXT2)
toks2 = tok.tokens(enc2['input_ids'])
print('tokens:', toks2)

# 假设模型预测 span = token 下标 1..3（"new york city"）
start_tok, end_tok = 1, 3
cs = enc2['offset_mapping'][start_tok][0]
ce = enc2['offset_mapping'][end_tok][1]
answer_from_offsets = TEXT2[cs:ce]
answer_from_decode = ' '.join(toks2[start_tok:end_tok + 1])
print(f'\n用 offsets 从原文切  : {answer_from_offsets!r}   ← 逐字符来自原文 ✅')
print(f'用 tokens 拼接      : {answer_from_decode!r}   ← 大小写丢了 ❌')
assert answer_from_offsets == 'New York City', '大小写、空格都保持原样'
assert answer_from_decode != answer_from_offsets
print('\n⚠️  症状：「答案基本对但标点/空格/大小写有细微差异」-> EM 分数莫名偏低。')
print('✅ 抽取式 QA 的答案必须用 offset_mapping 从**原始字符串**切。')

def span_to_text(text, offsets, i, j):
    return text[offsets[i][0]:offsets[j][1]]
for i, j in [(1, 1), (1, 2), (5, 5), (5, 7)]:
    print(f'  token[{i}..{j}] -> {span_to_text(TEXT2, enc2["offset_mapping"], i, j)!r}')
assert span_to_text(TEXT2, enc2['offset_mapping'], 5, 5) in TEXT2

## 2 · padding / truncation：算力浪费与静默截断

$$\text{token 浪费} = 1 - \frac{\sum L_i}{B\max L_i},\qquad
\text{注意力浪费} = 1 - \frac{\sum L_i^2}{B(\max L_i)^2}$$

**注意力是平方的，所以长度不齐时它的浪费被放大。**

In [ ]:
def waste_rates(lengths, pad_to=None):
    B = len(lengths); target = pad_to or max(lengths)
    tok_used = sum(lengths); tok_total = B * target
    att_used = sum(l * l for l in lengths); att_total = B * target * target
    return 1 - tok_used / tok_total, 1 - att_used / att_total

batch = [60, 80, 55, 500]
print(f"{'策略':<34s} {'补到':>6s} {'token浪费':>10s} {'注意力浪费':>11s}")
for label, pad_to in [("padding='max_length' (512)", 512), ("padding=True (动态)", None)]:
    tw, aw = waste_rates(batch, pad_to)
    print(f'{label:<34s} {pad_to or max(batch):>6d} {tw:>10.1%} {aw:>11.1%}')

# 按长度分组：把长度接近的放一批
all_lens = [60, 80, 55, 500, 62, 75, 58, 490]
def grouped_waste(lengths, bs=4):
    s = sorted(lengths); tws, aws = [], []
    for i in range(0, len(s), bs):
        g = s[i:i + bs]
        tw, aw = waste_rates(g)
        tws.append(tw); aws.append(aw)
    return float(np.mean(tws)), float(np.mean(aws))

tw_dyn, aw_dyn = waste_rates(all_lens)
tw_grp, aw_grp = grouped_waste(all_lens)
print(f'\n8 条混合长度: 动态 padding    token浪费 {tw_dyn:.1%} / 注意力浪费 {aw_dyn:.1%}')
print(f'              按长度分组     token浪费 {tw_grp:.1%} / 注意力浪费 {aw_grp:.1%}')
assert aw_grp < aw_dyn / 2, '按长度分组应把注意力浪费减半以上'
tw512, aw512 = waste_rates(batch, 512)
assert aw512 > 0.70, "padding='max_length' 在这个 batch 上浪费 70%+ 的注意力计算"
print(f'\n✅ 真正的杠杆是**按长度分组**（Trainer 的 group_by_length=True），')
print(f'   而不只是动态 padding。这里注意力浪费从 {aw_dyn:.0%} 降到 {aw_grp:.0%}。')

In [ ]:
def truncation_stats(lengths, max_length):
    '''截断率与信息损失率必须**分开看**。'''
    n_trunc = sum(1 for l in lengths if l > max_length)
    lost = sum(max(0, l - max_length) for l in lengths)
    return {'截断率': n_trunc / len(lengths),
            '信息损失率': lost / sum(lengths),
            'p50': int(np.percentile(lengths, 50)),
            'p95': int(np.percentile(lengths, 95)),
            'p99': int(np.percentile(lengths, 99)),
            'max': max(lengths)}

r = np.random.default_rng(5)
# 场景 A：多数样本略超（截断率高、损失率低）
lens_a = list(r.integers(120, 160, size=1000))
# 场景 B：少数极长样本（截断率低、损失率高）
lens_b = list(r.integers(30, 100, size=950)) + list(r.integers(2000, 4000, size=50))
MAXLEN = 128
for name, lens in [('A: 多数略超', lens_a), ('B: 少数极长', lens_b)]:
    s = truncation_stats(lens, MAXLEN)
    print(f'{name}: 截断率 {s["截断率"]:>6.1%} | 信息损失率 {s["信息损失率"]:>6.1%} | '
          f'p50={s["p50"]} p95={s["p95"]} p99={s["p99"]} max={s["max"]}')

sa, sb = truncation_stats(lens_a, MAXLEN), truncation_stats(lens_b, MAXLEN)
assert sa['截断率'] > sb['截断率'], 'A 的截断率更高'
assert sb['信息损失率'] > sa['信息损失率'], '但 B 的信息损失率更高'
print('\n⚠️  两个数要分开看：')
print('   A（截断率高、损失率低）通常可接受 —— 每条只丢一点。')
print('   B（截断率低、损失率高）意味着你在**系统性丢弃某个子群体** —— 比随机丢失更危险。')
print('✅ 建议：预处理时统计这两个数，把「信息损失率 > 5%」当作告警。')

## 3 · Dataset.map：批处理、指纹缓存与 remove_columns

In [ ]:
class MiniDataset:
    '''迷你 Dataset：列式存储 + map 的批处理语义 + 指纹缓存。'''
    def __init__(self, columns, _cache=None):
        self.columns = {k: list(v) for k, v in columns.items()}
        self._cache = {} if _cache is None else _cache
        self.map_calls = 0                      # 统计真正执行的次数（用于验证缓存）
    @property
    def column_names(self): return list(self.columns)
    def __len__(self): return len(next(iter(self.columns.values())))
    def __getitem__(self, i):
        if isinstance(i, int): return {k: v[i] for k, v in self.columns.items()}
        return MiniDataset({k: v[i] for k, v in self.columns.items()}, self._cache)
    def select(self, idxs):
        return MiniDataset({k: [v[i] for i in idxs] for k, v in self.columns.items()}, self._cache)

    def _fingerprint(self, fn, batched, batch_size, remove_columns, fn_kwargs):
        payload = json.dumps({
            'data': hashlib.sha1(json.dumps(self.columns, sort_keys=True,
                                            ensure_ascii=False).encode()).hexdigest(),
            # 函数字节码 + 常量表 —— 改函数体或改里面的常量，指纹都会变
            'fn': fn.__code__.co_code.hex() + repr(fn.__code__.co_consts),
            'batched': batched, 'bs': batch_size,
            'rm': sorted(remove_columns or []), 'kw': sorted((fn_kwargs or {}).items()),
        }, sort_keys=True)
        return hashlib.sha1(payload.encode()).hexdigest()[:12]

    def map(self, fn, batched=False, batch_size=1000, remove_columns=None,
            fn_kwargs=None, load_from_cache_file=True):
        fp = self._fingerprint(fn, batched, batch_size, remove_columns, fn_kwargs)
        if load_from_cache_file and fp in self._cache:
            return MiniDataset(self._cache[fp], self._cache)     # 缓存命中：不执行 fn
        self.map_calls += 1
        kw = fn_kwargs or {}
        keep = {k: v for k, v in self.columns.items() if k not in (remove_columns or [])}
        new = {}
        if batched:
            for i in range(0, len(self), batch_size):
                batch = {k: v[i:i + batch_size] for k, v in self.columns.items()}
                out = fn(batch, **kw)            # ← 收到的是 dict of **lists**
                for k, v in out.items(): new.setdefault(k, []).extend(v)
        else:
            for i in range(len(self)):
                out = fn(self[i], **kw)          # ← 收到的是 dict of **scalars**
                for k, v in out.items(): new.setdefault(k, []).append(v)
        merged = dict(keep); merged.update(new)
        lens = {k: len(v) for k, v in merged.items()}
        if len(set(lens.values())) > 1:
            raise ValueError(f'Column lengths mismatch: {lens} '
                             f'-> 你的 map 改变了行数，必须传 remove_columns=ds.column_names')
        self._cache[fp] = merged
        return MiniDataset(merged, self._cache)

TEXTS = ['Washington is nice', 'New York City is the capital of USA .',
         'the city is very good', 'Washington .']
ds = MiniDataset({'text': TEXTS, 'label': [1, 0, 1, 0]})
print(f'原始数据集: {len(ds)} 行, 列 = {ds.column_names}')
print('第 0 行:', ds[0])

In [ ]:
# batched=False vs True：**函数签名不同**，这是最常犯的错
def tok_single(example):
    e = tok(example['text'], max_length=12, truncation=True)
    return {'input_ids': e['input_ids'], 'n_tok': len(e['input_ids'])}

def tok_batched(batch):
    ids = [tok(t, max_length=12, truncation=True)['input_ids'] for t in batch['text']]
    return {'input_ids': ids, 'n_tok': [len(x) for x in ids]}

d1 = ds.map(tok_single, batched=False)
d2 = ds.map(tok_batched, batched=True, batch_size=2)
assert d1.columns['input_ids'] == d2.columns['input_ids'], '两种写法结果必须一致'
print(f'batched=False 与 batched=True 结果一致 ✅  n_tok = {d1.columns["n_tok"]}')

# 反例：把 batched=True 的函数当 batched=False 用
try:
    ds.map(tok_batched, batched=False)
    print('⚠️ 没报错，但结果是错的（tok 收到的是字符串而非列表）')
except Exception as e:
    print(f'✅ 签名不匹配报错: {type(e).__name__}')
print('\n记住：batched=False 时 example["text"] 是**字符串**；batched=True 时是**字符串列表**。')

In [ ]:
# 指纹缓存：同样的 (数据, 函数, 参数) -> 不重新执行
ds2 = MiniDataset({'text': TEXTS, 'label': [1, 0, 1, 0]})
before = ds2.map_calls
_ = ds2.map(tok_batched, batched=True)
after_first = ds2.map_calls
_ = ds2.map(tok_batched, batched=True)          # 第二次：应命中缓存
after_second = ds2.map_calls
print(f'实际执行次数: 首次 {after_first - before} | 再调一次 {after_second - after_first}')
assert after_first - before == 1 and after_second - after_first == 0, '第二次应命中缓存'
print('✅ 指纹命中 -> 不执行函数（真实库里是直接读缓存的 arrow 文件）')

# 改了函数体 -> 指纹变 -> 重新执行
def tok_batched_v2(batch):
    ids = [tok(t, max_length=8, truncation=True)['input_ids'] for t in batch['text']]
    return {'input_ids': ids, 'n_tok': [len(x) for x in ids]}
_ = ds2.map(tok_batched_v2, batched=True)
assert ds2.map_calls == after_second + 1, '改了函数体应重新执行'
print('✅ 改函数体 -> 字节码变 -> 指纹变 -> 重新计算')

# load_from_cache_file=False 强制重算（依赖外部可变状态时需要）
n = ds2.map_calls
_ = ds2.map(tok_batched, batched=True, load_from_cache_file=False)
assert ds2.map_calls == n + 1
print('✅ load_from_cache_file=False 强制重算（函数依赖全局变量/文件/随机数时用它）')

In [ ]:
# remove_columns：当 map **改变行数**时必需
def sliding_window(batch, window=6, stride=3):
    '''长文本切成多个重叠窗口 —— 行数会变多！'''
    out_ids, out_src = [], []
    for src_i, t in enumerate(batch['text']):
        ids = tok(t, add_special_tokens=False)['input_ids']
        for s in range(0, max(1, len(ids) - window + 1), stride):
            out_ids.append(ids[s:s + window]); out_src.append(src_i)
    return {'input_ids': out_ids, 'src': out_src}

try:
    ds.map(sliding_window, batched=True)
    raise RuntimeError('不该到这')
except ValueError as e:
    print(f'❌ 不删旧列: {str(e)[:110]}…')

d3 = ds.map(sliding_window, batched=True, remove_columns=ds.column_names)
print(f'\n✅ 删掉旧列后: {len(ds)} 行 -> {len(d3)} 行（滑窗产生更多样本）')
assert len(d3) > len(ds)
assert d3.column_names == ['input_ids', 'src']
print('   规则：**只要你的 map 可能改变行数，就必须 remove_columns=ds.column_names**')

## 4 · IterableDataset 与缓冲区打乱

流式只能顺序迭代、没有 `len()`、`shuffle` 只能靠**缓冲区近似**。
缓冲太小 + 数据在磁盘上有序 = 同 batch 样本高度相关，显著伤害训练。

In [ ]:
class MiniIterableDataset:
    def __init__(self, gen_fn): self.gen_fn = gen_fn
    def __iter__(self): return self.gen_fn()
    def map(self, fn):
        outer = self.gen_fn
        return MiniIterableDataset(lambda: (fn(x) for x in outer()))    # 惰性
    def shuffle(self, buffer_size, seed=0):
        outer = self.gen_fn
        def gen():
            r = random.Random(seed); buf = []
            for x in outer():
                buf.append(x)
                if len(buf) >= buffer_size:
                    j = r.randrange(len(buf)); yield buf.pop(j)
            r.shuffle(buf)
            yield from buf
        return MiniIterableDataset(gen)
    def take(self, n):
        outer = self.gen_fn
        def gen():
            for i, x in enumerate(outer()):
                if i >= n: return
                yield x
        return MiniIterableDataset(gen)

# 磁盘上按「来源」有序排列：前 500 条来自源 A，后 500 条来自源 B
def sorted_stream():
    for i in range(1000):
        yield {'src': 'A' if i < 500 else 'B', 'i': i}

stream = MiniIterableDataset(sorted_stream)
try:
    len(stream); raise RuntimeError('不该到这')
except TypeError:
    print('✅ IterableDataset 没有 len() —— 只能顺序迭代')

def batch_purity(ds, batch=16, n_batches=20):
    '''每个 batch 里「同一来源」的占比。1.0 = 完全同源（最差）。'''
    it, purities, cur = iter(ds), [], []
    for x in it:
        cur.append(x['src'])
        if len(cur) == batch:
            purities.append(max(cur.count('A'), cur.count('B')) / batch)
            cur = []
            if len(purities) >= n_batches: break
    return float(np.mean(purities))

print(f"\n{'buffer_size':>12s} {'batch 内同源占比':>16s}")
for bs in [1, 16, 128, 1024]:
    pur = batch_purity(stream.shuffle(buffer_size=bs, seed=1))
    print(f'{bs:>12d} {pur:>16.2f}')

p_small = batch_purity(stream.shuffle(buffer_size=1, seed=1))
p_large = batch_purity(stream.shuffle(buffer_size=1024, seed=1))
assert p_small > 0.95, '缓冲=1 等于不打乱 -> 每个 batch 完全同源'
assert p_large < p_small, '缓冲越大越接近全局打乱'
print(f'\n✅ 缓冲 1 时同源占比 {p_small:.2f}（完全没打乱）；缓冲 1024 时 {p_large:.2f}。')
print('   若数据在磁盘上按来源/时间有序，小缓冲会让同 batch 样本高度相关 —— 显著伤害训练。')
print('   真实库的流式 shuffle 会**同时打乱 shard 顺序**，这是两级打乱的第一级。')
print('   「缓冲要多大才够」的完整量化见 C43 模块 02。')

## 5 · DataCollator：-100 的构造与「打印一个 batch」检查器

`-100` 是全生态约定（`CrossEntropyLoss(ignore_index=-100)`）：
padding、未被掩的位置、SFT 里的 prompt 部分——**都要填 -100**。漏填会让模型学错东西。

In [ ]:
PAD_ID = STOI[PAD]

def collate_with_padding(features, pad_id=PAD_ID):
    '''对应 DataCollatorWithPadding：动态补齐 + attention_mask。'''
    L = max(len(f['input_ids']) for f in features)
    return {
        'input_ids': np.array([f['input_ids'] + [pad_id] * (L - len(f['input_ids'])) for f in features]),
        'attention_mask': np.array([[1]*len(f['input_ids']) + [0]*(L-len(f['input_ids'])) for f in features]),
        'labels': np.array([f['label'] for f in features]),
    }

def collate_token_classification(features, pad_id=PAD_ID):
    '''对应 DataCollatorForTokenClassification：labels 也要补齐，用 **-100**。'''
    L = max(len(f['input_ids']) for f in features)
    return {
        'input_ids': np.array([f['input_ids'] + [pad_id]*(L-len(f['input_ids'])) for f in features]),
        'attention_mask': np.array([[1]*len(f['input_ids']) + [0]*(L-len(f['input_ids'])) for f in features]),
        'labels': np.array([f['labels'] + [-100]*(L-len(f['labels'])) for f in features]),
    }

def collate_clm(features, pad_id=PAD_ID):
    '''对应 DataCollatorForLanguageModeling(mlm=False)：labels=input_ids，pad 位置置 -100。'''
    L = max(len(f['input_ids']) for f in features)
    ids = np.array([f['input_ids'] + [pad_id]*(L-len(f['input_ids'])) for f in features])
    labels = ids.copy(); labels[ids == pad_id] = -100
    return {'input_ids': ids, 'attention_mask': (ids != pad_id).astype(int), 'labels': labels}

feats = [{'input_ids': tok(t)['input_ids'], 'label': i % 2} for i, t in enumerate(TEXTS)]
b = collate_with_padding(feats)
print('分类 batch 形状:', {k: v.shape for k, v in b.items()})
assert b['input_ids'].shape[0] == len(feats)
assert (b['attention_mask'].sum(1) == [len(f['input_ids']) for f in feats]).all()

feats_tc = [{'input_ids': tok(t)['input_ids'],
             'labels': align_word_labels(tok(t)['_word_ids'],
                                         [L2I['O']] * 20)} for t in TEXTS]
btc = collate_token_classification(feats_tc)
assert (btc['labels'] == -100).sum() > 0, 'padding 与非首子词位置必须是 -100'
bclm = collate_clm(feats)
assert (bclm['labels'][bclm['input_ids'] == PAD_ID] == -100).all(), 'CLM: pad 位置必须 -100'
print('✅ 三种 collator 的 -100 处理正确')

# 反例：忘记把 pad 位置置 -100
bad_labels = np.array([f['input_ids'] + [PAD_ID]*(bclm['input_ids'].shape[1]-len(f['input_ids']))
                       for f in feats])
n_pad = int((bad_labels == PAD_ID).sum())
print(f'\n❌ 忘记置 -100: 有 {n_pad} 个位置在教模型「预测 [PAD]」')
print('   pad 极易预测 -> loss 看起来更低（甚至下降更快）-> 但效果变差。')
assert n_pad > 0

In [ ]:
def inspect_batch(batch, tokenizer, max_rows=2):
    '''训练开始前必做的四项检查。三十秒发现 80% 的数据侧 bug。'''
    print('=' * 68)
    print('① 形状:', {k: tuple(v.shape) for k, v in batch.items()})
    ids, am = batch['input_ids'], batch.get('attention_mask')
    labels = batch.get('labels')
    for r in range(min(max_rows, ids.shape[0])):
        n_real = int(am[r].sum()) if am is not None else ids.shape[1]
        print(f'\n② 第 {r} 行解码（前 {n_real} 个真实 token）:')
        print('   ', ' '.join(tokenizer.tokens(list(ids[r][:n_real]))))
        if am is not None:
            zeros = np.where(am[r] == 0)[0]
            ok = len(zeros) == 0 or (zeros[0] == n_real and (np.diff(zeros) == 1).all())
            print(f'③ attention_mask 的 0 是否只在尾部连续: {"✅" if ok else "❌"}')
        if labels is not None and labels.ndim == 2:
            frac = float((labels[r] == -100).mean())
            print(f'④ labels 中 -100 占比: {frac:.1%}')
    print('=' * 68)
    if labels is not None and labels.ndim == 2:
        overall = float((labels == -100).mean())
        assert overall < 0.95, f'-100 占比 {overall:.1%} 过高 —— 几乎没有训练信号！'
    return True

assert inspect_batch(bclm, tok)
print('\n✅ 把这个函数放在训练循环前跑一次 —— 它检查形状、解码通顺性、mask 位置、-100 比例。')

## ✏️ 练习 1：sequence_ids —— 区分特殊 token / 句 A / 句 B

实现 `sequence_ids(word_ids, sep_positions)`：返回同长度列表，
特殊 token 位置为 `None`，属于第一句的为 `0`，第二句的为 `1`。
`sep_positions` 是所有特殊 token 的下标集合；第一个特殊 token 之后到第一个 SEP 之前是句 A，
第一个 SEP 之后到最后一个 SEP 之前是句 B。

简化规则：遍历，遇到特殊 token 填 `None` 并把「当前句号」在**遇到第一个 SEP 之后**加一。

In [ ]:
def sequence_ids(word_ids, sep_positions):
    # TODO: 遍历 word_ids；下标在 sep_positions 里 -> None（且若不是第 0 个特殊 token 则句号+1）
    #       否则填当前句号
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
#  [CLS] a b [SEP] c d [SEP]
wids = [None, 0, 1, None, 2, 3, None]
seps = {0, 3, 6}
out = sequence_ids(wids, seps)
assert out == [None, 0, 0, None, 1, 1, None], out
# 单句：[CLS] a b [SEP]
assert sequence_ids([None, 0, 1, None], {0, 3}) == [None, 0, 0, None]
# 全特殊
assert sequence_ids([None, None], {0, 1}) == [None, None]
print('sequence_ids:', out)
print('✅ 练习 1 通过：QA 任务用它把 span 搜索限制在**上下文那一句**（C49 模块 03）')

## ✏️ 练习 2：按长度分组的 batch 采样器

实现 `length_grouped_batches(lengths, batch_size, mega_batch_mult=4, seed=0)`：
Trainer 的 `group_by_length` 就是这个思路——
① 把索引按 `mega_batch = batch_size * mega_batch_mult` 分块（块内先随机）；
② 每块内按长度排序；③ 切成 batch；④ 最后把所有 batch 的顺序打乱（保留随机性）。
返回 batch 的索引列表（`List[List[int]]`）。

In [ ]:
def length_grouped_batches(lengths, batch_size, mega_batch_mult=4, seed=0):
    # TODO: 见上述四步
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
lens = list(np.random.default_rng(2).integers(20, 500, size=64))
batches = length_grouped_batches(lens, batch_size=8, mega_batch_mult=4, seed=1)
flat = sorted(i for b in batches for i in b)
assert flat == list(range(64)), '必须覆盖每个样本恰好一次'
assert all(len(b) <= 8 for b in batches)
# 分组后 batch 内长度应比随机分组更接近
def mean_pad_waste(bs):
    return float(np.mean([waste_rates([lens[i] for i in b])[0] for b in bs]))
rnd = np.random.default_rng(3).permutation(64)
random_batches = [list(rnd[i:i+8]) for i in range(0, 64, 8)]
w_grp, w_rnd = mean_pad_waste(batches), mean_pad_waste(random_batches)
print(f'按长度分组的平均 padding 浪费: {w_grp:.1%}')
print(f'随机分组的平均 padding 浪费  : {w_rnd:.1%}')
assert w_grp < w_rnd * 0.7, '分组应显著降低浪费'
print('✅ 练习 2 通过：这就是 Trainer 的 group_by_length=True（代价是牺牲一点随机性）')

## ✏️ 练习 3：SFT 的 prompt 掩码

SFT 时只应对**回答部分**计损失，prompt 部分要填 `-100`（否则模型在背诵 prompt）。
实现 `mask_prompt_labels(input_ids, prompt_len, pad_id)`：
返回 labels —— 前 `prompt_len` 个位置为 `-100`，pad 位置为 `-100`，其余等于 `input_ids`。

In [ ]:
def mask_prompt_labels(input_ids, prompt_len, pad_id):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
ids = np.array([1, 5, 6, 7, 8, 9, PAD_ID, PAD_ID])
labs = mask_prompt_labels(ids, prompt_len=3, pad_id=PAD_ID)
assert labs.tolist() == [-100, -100, -100, 7, 8, 9, -100, -100], labs.tolist()
n_signal = int((labs != -100).sum())
assert n_signal == 3, f'只有回答的 3 个 token 计损失，得到 {n_signal}'
# prompt 很长时几乎没有信号 -> 应该被 inspect_batch 的断言抓住
labs2 = mask_prompt_labels(np.arange(1, 21), prompt_len=19, pad_id=PAD_ID)
assert float((labs2 == -100).mean()) > 0.9
print(f'labels = {labs.tolist()}  (只有回答部分计损失)')
print('✅ 练习 3 通过：**忘记掩 prompt 是 SFT 最经典的 bug** —— loss 好看但模型在背诵输入')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def sequence_ids(word_ids, sep_positions):
    out, seq, seen_special = [], 0, 0
    for i, w in enumerate(word_ids):
        if i in sep_positions:
            out.append(None)
            if seen_special > 0:      # 第一个特殊 token（CLS）之后才开始计句号
                seq += 1
            seen_special += 1
        else:
            out.append(seq)
    return out

In [ ]:
# 练习 2 参考答案
def length_grouped_batches(lengths, batch_size, mega_batch_mult=4, seed=0):
    r = np.random.default_rng(seed)
    idx = list(r.permutation(len(lengths)))
    mega = batch_size * mega_batch_mult
    batches = []
    for i in range(0, len(idx), mega):
        chunk = sorted(idx[i:i + mega], key=lambda j: lengths[j])
        for k in range(0, len(chunk), batch_size):
            batches.append(chunk[k:k + batch_size])
    order = r.permutation(len(batches))
    return [batches[i] for i in order]

In [ ]:
# 练习 3 参考答案
def mask_prompt_labels(input_ids, prompt_len, pad_id):
    labels = np.array(input_ids).copy()
    labels[:prompt_len] = -100
    labels[np.array(input_ids) == pad_id] = -100
    return labels

---
## 🧪 真实 API 对照胶囊（不在本环境运行，可原样复制）

In [ ]:
RECIPE = r'''
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorForTokenClassification

CKPT = "microsoft/deberta-v3-base"
tok = AutoTokenizer.from_pretrained(CKPT, use_fast=True)   # ① 必须 fast，否则没有 word_ids

ds = load_dataset("conll2003")            # 大数据集加 streaming=True

def prepare(batch):
    enc = tok(batch["tokens"],
              is_split_into_words=True,    # ② 输入已是词列表
              truncation=True, max_length=256,
              return_offsets_mapping=False)
    labels = []
    for i, tags in enumerate(batch["ner_tags"]):
        wids, prev, row = enc.word_ids(batch_index=i), None, []
        for w in wids:
            if w is None:      row.append(-100)      # ③ 特殊 token
            elif w != prev:    row.append(tags[w])   # ④ 每个词的第一个子词
            else:              row.append(-100)      # ⑤ 后续子词不计损失
            prev = w
        labels.append(row)
    enc["labels"] = labels
    return enc

# ⑥ batched=True 快 10-100 倍；⑦ 改变列 -> 必须 remove_columns
tokenized = ds.map(prepare, batched=True, batch_size=1000,
                   remove_columns=ds["train"].column_names)

# ⑧ 动态 padding（labels 用 -100 补齐）
collator = DataCollatorForTokenClassification(tok, padding=True)

# ⑨ 训练前必做：打印一个 batch 检查四件事
import torch
batch = collator([tokenized["train"][i] for i in range(4)])
print({k: tuple(v.shape) for k, v in batch.items()})
print(tok.decode(batch["input_ids"][0]))
print("labels -100 占比:", (batch["labels"] == -100).float().mean().item())

# ⑩ 统计截断率（放进预处理脚本，超阈值就告警）
lens = [len(tok(" ".join(x["tokens"]))["input_ids"]) for x in ds["train"].select(range(2000))]
import numpy as np
print("p50/p95/p99:", np.percentile(lens, [50, 95, 99]),
      "截断率:", np.mean(np.array(lens) > 256))
'''
print(RECIPE)
for c in ['use_fast=True', 'word_ids', '-100', 'batched=True', 'remove_columns', '截断率']:
    assert c in RECIPE, c
print('✅ 配方覆盖本模块全部要点')

### 小结
- **fast tokenizer 的真正价值是 `offset_mapping` 与 `word_ids()`**：前者把 token 预测映射回原文字符（QA 必需，**不能用 decode 拼**），后者把词级标注对齐到子词（NER 必需）。特殊 token 的 offset 是 `(0,0)` 不是 `None`。
- **`padding='max_length'` 可能浪费 89% 的注意力计算**；动态 padding 帮助有限，真正的杠杆是 **按长度分组**（`group_by_length=True`）。
- **截断是静默的**。要分开统计**截断率**与**信息损失率**——后者高意味着系统性丢弃某个子群体。
- **`map(batched=True)` 快 10–100 倍，但函数签名不同**（字符串 vs 字符串列表）；**改变行数就必须 `remove_columns`**；指纹缓存按函数字节码算，依赖外部可变状态时要 `load_from_cache_file=False`。
- **流式没有 `len()`、只能缓冲区近似打乱**；缓冲太小 + 磁盘有序 = 同 batch 高度相关。
- **`-100` 是全生态约定**：padding、非首子词、SFT 的 prompt 部分都要填它。漏填让 loss 更好看但效果更差。
- 带走那个 **「打印一个 batch 检查四件事」** 的习惯：形状、解码通顺性、mask 位置、-100 比例。

下一站：**模块 03 · Trainer 与 TrainingArguments** —— 训练循环里那些相乘才有意义的参数。